# NPT Equilibration with DPA4C

## Configurations

> Configurations were equilibrated under NPT at 300 K and 1 atm using the DPA4C potential (DeePMD-kit), trained on OMat24 and OMol25.

---

DPA4C is a compact equivariant potential co-designed with compressed CUDA operators. It approaches MACE-Omat accuracy at 100x higher throughput. Five variants span a 49-fold parameter range:

| Variant | Params | Throughput (M atoms/s) | Use case |
|---------|--------|----------------------|----------|
| DPA4C-Nano | 0.030M | 16.5 | Fastest, large systems |
| DPA4C-Mini | 0.146M | 10.2 | Fast, good accuracy |
| DPA4C-Neo | 0.342M | 7.0 | Balanced |
| DPA4C-Air | 0.434M | 3.8 | High accuracy |
| DPA4C-Plus | 1.457M | 2.2 | Highest accuracy |

---

## How to use this

1. Run **Section 1** once per session
2. Set `AUTO_STOP = True` if you want the run to stop itself when density converges (recommended)
3. **Section 2** to start or resume NPT
4. **Section 3** for status and download

If runtime disconnects, reconnect, run Section 1, then use 2c to resume.

---
## 1.) Installs and Setup

In [ ]:
!pip install -q ase
!pip install -q deepmd-kit[torch]

In [ ]:
# Download the DPA4C model checkpoint
# Change the variant here: Nano, Mini, Neo, Air, Plus
DPA4C_VARIANT = 'Mini'  # <-- change this to pick a different size

import subprocess
result = subprocess.run(['dp', 'pretrained', 'download', f'DPA4C-{DPA4C_VARIANT}'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(f'Download failed. stderr: {result.stderr}')
    print('Try listing available models with: !dp pretrained list')

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.md.melchionna import MelchionnaNPT as NPT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
import torch
from deepmd.calculator import DP

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

DRIVE_ROOT = '/content/gdrive/MyDrive/DPA4C_equilibration'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────

NPT_STEPS = 100000        # max steps (100 ps) — only matters if AUTO_STOP is off
AUTO_STOP = True          # stop automatically when density converges
CONVERGENCE_PCT = 1.0     # drift threshold for convergence (percent)
CHECK_EVERY = 10000       # check convergence every N steps
CHECKPOINT_EVERY = 1000   # save checkpoint every N steps
LOG_EVERY = 100           # log T, PE, rho every N steps

# Find the downloaded model file
MODEL_PATH = None
for ext in ['*.pth', '*.pt', '*.pb']:
    matches = glob.glob(f'*DPA4C*{ext}') + glob.glob(f'*dpa4c*{ext}')
    if matches:
        MODEL_PATH = matches[0]
        break
if MODEL_PATH is None:
    # try the dp pretrained cache
    for ext in ['*.pth', '*.pt']:
        matches = glob.glob(os.path.expanduser(f'~/.cache/deepmd/**/DPA4C*{ext}'), recursive=True)
        if matches:
            MODEL_PATH = matches[0]
            break
if MODEL_PATH:
    print(f'Model: {MODEL_PATH}')
else:
    print('WARNING: No DPA4C model file found. Run the download cell above.')
    print('Or set MODEL_PATH manually below.')
    # MODEL_PATH = '/path/to/your/dpa4c_model.pth'  # uncomment and set manually


# ── Shared functions ──────────────────────────────────────────────────────

def get_calculator():
    return DP(model=MODEL_PATH)


def box_dir(name):
    d = os.path.join(DRIVE_ROOT, name)
    os.makedirs(d, exist_ok=True)
    return d


def save_checkpoint(atoms, name, step, temps, pes, densities):
    d = box_dir(name)
    for f in glob.glob(os.path.join(d, 'ckpt_*.xyz')):
        os.remove(f)
    for f in glob.glob(os.path.join(d, 'ckpt_*.npz')):
        os.remove(f)
    write(os.path.join(d, f'ckpt_{step}.xyz'), atoms, format='extxyz')
    np.savez(os.path.join(d, f'ckpt_{step}.npz'),
             step=step, temps=np.array(temps),
             pes=np.array(pes), densities=np.array(densities))
    rho = densities[-1] if densities else 0
    print(f'    [CHECKPOINT] step {step}, rho={rho:.4f} g/cm3', flush=True)


def load_checkpoint(name):
    d = box_dir(name)
    xyzs = sorted(glob.glob(os.path.join(d, 'ckpt_*.xyz')))
    npzs = sorted(glob.glob(os.path.join(d, 'ckpt_*.npz')))
    if not xyzs or not npzs:
        return None, 0, [], [], []
    atoms = read(xyzs[-1])
    data = np.load(npzs[-1])
    return (atoms, int(data['step']),
            data['temps'].tolist(), data['pes'].tolist(), data['densities'].tolist())


def check_convergence(densities, threshold_pct=CONVERGENCE_PCT):
    n = len(densities)
    if n < 100:
        return False, float('inf'), 0.0
    window = n // 5
    prev = densities[-(2*window):-window]
    last = densities[-window:]
    mean_prev = np.mean(prev)
    mean_last = np.mean(last)
    drift = 100 * abs(mean_last - mean_prev) / mean_last
    return drift < threshold_pct, drift, mean_last


def run_npt(atoms, name, n_steps=NPT_STEPS, start_step=0,
            prev_temps=None, prev_pes=None, prev_densities=None):
    atoms.pbc = True
    total_mass = sum(atoms.get_masses())

    torch.cuda.empty_cache()
    atoms.calc = get_calculator()

    if start_step == 0:
        MaxwellBoltzmannDistribution(atoms, temperature_K=300.0)

    dyn = NPT(atoms, timestep=1.0*units.fs, temperature_K=300.0,
              externalstress=1.0*units.bar,
              ttime=100*units.fs,
              pfactor=0.1,
              mask=[[1,0,0],[0,1,0],[0,0,1]])
    dyn.nsteps = start_step

    temps = list(prev_temps or [])
    pes = list(prev_pes or [])
    densities = list(prev_densities or [])
    remaining = n_steps - start_step
    t0 = time.time()
    converged_at = None

    rho0 = total_mass / atoms.get_volume() * 1.66054
    print(f'  {len(atoms)} atoms | starting rho={rho0:.4f} g/cm3 | {remaining} steps remaining', flush=True)
    if AUTO_STOP:
        print(f'  AUTO_STOP enabled: will stop when density drift < {CONVERGENCE_PCT}%', flush=True)

    def logger():
        T = atoms.get_temperature()
        pe = atoms.get_potential_energy() / len(atoms)
        rho = total_mass / atoms.get_volume() * 1.66054
        temps.append(T); pes.append(pe); densities.append(rho)
        step = dyn.nsteps
        if step % 1000 == 0:
            elapsed = time.time() - t0
            rate = (step - start_step) / elapsed if elapsed > 0 else 0
            eta = (n_steps - step) / rate / 60 if rate > 0 else 0
            conv_str = ''
            converged, drift, mean_rho = check_convergence(densities)
            if converged:
                conv_str = f' ** CONVERGED (drift={drift:.2f}%, mean={mean_rho:.4f}) **'
            elif drift < float('inf'):
                conv_str = f' (drift={drift:.1f}%)'
            print(f'    step {step:6d}/{n_steps}  T={T:.1f}K  rho={rho:.4f}  '
                  f'PE={pe:.4f}  [{rate:.1f} st/s, ETA {eta:.0f}m]{conv_str}', flush=True)

    def checkpointer():
        step = dyn.nsteps
        if step > start_step and step % CHECKPOINT_EVERY == 0:
            save_checkpoint(atoms, name, step, temps, pes, densities)

    def convergence_checker():
        nonlocal converged_at
        if not AUTO_STOP or converged_at is not None:
            return
        step = dyn.nsteps
        if step > start_step and step % CHECK_EVERY == 0:
            converged, drift, mean_rho = check_convergence(densities)
            if converged:
                converged_at = step
                print(f'\n    >>> AUTO_STOP: Density converged at step {step} '
                      f'(drift={drift:.2f}%, mean={mean_rho:.4f} g/cm3) <<<', flush=True)
                raise StopIteration()

    dyn.attach(logger, interval=LOG_EVERY)
    dyn.attach(checkpointer, interval=LOG_EVERY)
    dyn.attach(convergence_checker, interval=LOG_EVERY)

    if remaining <= 0:
        print(f'  Already complete.', flush=True)
        return atoms, temps, pes, densities

    try:
        dyn.run(remaining)
    except StopIteration:
        pass  # auto-stop triggered

    final_step = converged_at or n_steps
    save_checkpoint(atoms, name, final_step, temps, pes, densities)
    final_path = os.path.join(box_dir(name), f'{name}.xyz')
    write(final_path, atoms, format='extxyz')

    rho_f = total_mass / atoms.get_volume() * 1.66054
    has_nan = np.any(np.isnan(atoms.get_positions()))
    if has_nan:
        print(f'  WARNING: NaN in final positions!', flush=True)
    else:
        converged, drift, mean_rho = check_convergence(densities)
        status = 'CONVERGED' if converged else f'NOT CONVERGED (drift={drift:.1f}%)'
        stopped = f' (auto-stopped at step {converged_at})' if converged_at else ''
        print(f'  DONE: rho {rho0:.4f} -> {rho_f:.4f} g/cm3 | {status}{stopped}', flush=True)
        print(f'  Final: {final_path}', flush=True)

    return atoms, temps, pes, densities


def plot_diagnostics(name, temps, pes, densities):
    if len(temps) == 0:
        print('No data to plot.')
        return
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))
    t_ps = np.arange(len(temps)) * LOG_EVERY * 0.001
    w = min(50, max(1, len(temps) // 4))

    ax1.plot(t_ps, temps, alpha=0.3, lw=0.5)
    if w > 1: ax1.plot(t_ps[w-1:], np.convolve(temps, np.ones(w)/w, 'valid'), 'r', lw=1.5)
    ax1.axhline(300, color='k', ls='--', alpha=0.4)
    ax1.set(xlabel='Time (ps)', ylabel='Temperature (K)', title='Temperature')

    ax2.plot(t_ps, pes, alpha=0.3, lw=0.5)
    if w > 1: ax2.plot(t_ps[w-1:], np.convolve(pes, np.ones(w)/w, 'valid'), 'r', lw=1.5)
    ax2.set(xlabel='Time (ps)', ylabel='PE (eV/atom)', title='Potential Energy')

    t_rho = np.arange(len(densities)) * LOG_EVERY * 0.001
    ax3.plot(t_rho, densities, alpha=0.3, lw=0.5)
    if w > 1: ax3.plot(t_rho[w-1:], np.convolve(densities, np.ones(w)/w, 'valid'), 'r', lw=1.5)
    converged, drift, mean_rho = check_convergence(densities)
    status = f'CONVERGED (rho={mean_rho:.4f})' if converged else f'drift={drift:.1f}%'
    ax3.set(xlabel='Time (ps)', ylabel='Density (g/cm3)', title=f'Density -- {status}')

    plt.suptitle(f'{name} (DPA4C-{DPA4C_VARIANT})', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(box_dir(name), 'diagnostics.png'), dpi=150)
    plt.show()


print(f'Setup complete. DPA4C-{DPA4C_VARIANT} ready.')
print(f'AUTO_STOP: {AUTO_STOP} (convergence threshold: {CONVERGENCE_PCT}%)')
print(f'Checkpoints every {CHECKPOINT_EVERY} steps, convergence check every {CHECK_EVERY} steps')

---
## 2.) NPT Equilibration
---

### 2a. Start NPT from scratch (upload a box)

In [ ]:
from google.colab import files
print('Upload a .xyz file to start NPT:')
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
BOX_NAME = INPUT_FILE.replace('_opt.xyz', '').replace('.xyz', '')
print(f'\nBox: {BOX_NAME}')
print(f'Checkpoints: {DRIVE_ROOT}/{BOX_NAME}/')

In [ ]:
atoms = read(INPUT_FILE)
print(f'Loaded: {len(atoms)} atoms, cell = {atoms.cell.lengths()}')

print(f'\n{"="*60}')
print(f'  NPT (DPA4C-{DPA4C_VARIANT}): {BOX_NAME}')
print(f'{"="*60}')
atoms, temps, pes, densities = run_npt(atoms, BOX_NAME)
plot_diagnostics(BOX_NAME, temps, pes, densities)

### 2b. Start NPT from a box on Drive

In [ ]:
drive_boxes = sorted([f.replace('.xyz','') for f in os.listdir(DRIVE_ROOT)
                       if f.endswith('.xyz') and not f.startswith('ckpt_')])
for d in sorted(os.listdir(DRIVE_ROOT)):
    dd = os.path.join(DRIVE_ROOT, d)
    if os.path.isdir(dd):
        final = os.path.join(dd, f'{d}.xyz')
        if not os.path.exists(final):
            drive_boxes.append(d + ' (not finished)')

print('Boxes on Drive:')
for i, b in enumerate(drive_boxes):
    print(f'  [{i}] {b}')
print('\nSet BOX_INDEX below.')

In [ ]:
BOX_INDEX = 0  # <-- change this
BOX_NAME = drive_boxes[BOX_INDEX].split(' (')[0]

candidates = [os.path.join(DRIVE_ROOT, f'{BOX_NAME}.xyz'),
              os.path.join(DRIVE_ROOT, BOX_NAME, f'{BOX_NAME}_input.xyz')]
INPUT_FILE = next((c for c in candidates if os.path.exists(c)), None)
if INPUT_FILE is None:
    print(f'No input for {BOX_NAME}. Upload with 2a.')
else:
    atoms = read(INPUT_FILE)
    print(f'\n{"="*60}')
    print(f'  NPT (DPA4C-{DPA4C_VARIANT}): {BOX_NAME}')
    print(f'{"="*60}')
    atoms, temps, pes, densities = run_npt(atoms, BOX_NAME)
    plot_diagnostics(BOX_NAME, temps, pes, densities)

### 2c. Resume from checkpoint

In [ ]:
print('Boxes on Drive:\n')
all_dirs = sorted([d for d in os.listdir(DRIVE_ROOT) if os.path.isdir(os.path.join(DRIVE_ROOT, d))])
for i, d in enumerate(all_dirs):
    dd = os.path.join(DRIVE_ROOT, d)
    final = os.path.exists(os.path.join(dd, f'{d}.xyz'))
    ckpts = sorted(glob.glob(os.path.join(dd, 'ckpt_*.npz')))
    if final:
        status = 'FINISHED'
    elif ckpts:
        data = np.load(ckpts[-1])
        status = f'step {int(data["step"])}/{NPT_STEPS}'
    else:
        status = 'no checkpoint'
    print(f'  [{i}] {d:<50s} {status}')
print('\nSet RESUME_INDEX below.')

In [ ]:
RESUME_INDEX = 0  # <-- change this

resume_name = all_dirs[RESUME_INDEX]
atoms, step, prev_t, prev_pe, prev_rho = load_checkpoint(resume_name)

if atoms is None:
    print(f'No checkpoint for {resume_name}. Use 2a.')
elif os.path.exists(os.path.join(box_dir(resume_name), f'{resume_name}.xyz')):
    print(f'Already complete! Use Section 3 to download.')
else:
    print(f'Resuming: {resume_name} from step {step}')
    atoms.pbc = True
    atoms, temps, pes, densities = run_npt(
        atoms, resume_name, start_step=step,
        prev_temps=prev_t, prev_pes=prev_pe, prev_densities=prev_rho)
    plot_diagnostics(resume_name, temps, pes, densities)

---
## 3.) Status and Download

In [ ]:
print(f'{"Box":<50s} {"Status":<15s} {"Final rho":<12s} {"Atoms"}')
print('-' * 90)
finished = []
for d in sorted(os.listdir(DRIVE_ROOT)):
    dd = os.path.join(DRIVE_ROOT, d)
    if not os.path.isdir(dd): continue
    final = os.path.join(dd, f'{d}.xyz')
    if os.path.exists(final):
        a = read(final)
        has_nan = np.any(np.isnan(a.get_positions()))
        if has_nan:
            status, rho_str = 'NaN!', 'N/A'
        else:
            rho = sum(a.get_masses()) / a.get_volume() * 1.66054
            status, rho_str = 'DONE', f'{rho:.4f} g/cm3'
            finished.append((d, final, rho))
        print(f'{d:<50s} {status:<15s} {rho_str:<12s} {len(a)}')
    else:
        ckpts = sorted(glob.glob(os.path.join(dd, 'ckpt_*.npz')))
        if ckpts:
            data = np.load(ckpts[-1])
            rho = data['densities'][-1] if len(data['densities']) > 0 else 0
            status = f'step {int(data["step"])}'
            print(f'{d:<50s} {status:<15s} {rho:.4f} g/cm3' if rho else f'{d:<50s} {status:<15s}')
        else:
            print(f'{d:<50s} {"empty":<15s}')

print(f'\n{len(finished)} boxes finished.')

In [ ]:
import shutil

if finished:
    dl_dir = '/content/npt_finished'
    os.makedirs(dl_dir, exist_ok=True)
    for name, path, rho in finished:
        shutil.copy(path, os.path.join(dl_dir, f'{name}.xyz'))
        diag = os.path.join(os.path.dirname(path), 'diagnostics.png')
        if os.path.exists(diag):
            shutil.copy(diag, os.path.join(dl_dir, f'{name}_diagnostics.png'))
    shutil.make_archive('/content/npt_finished', 'zip', dl_dir)
    from google.colab import files
    files.download('/content/npt_finished.zip')
    print(f'Downloaded {len(finished)} boxes as npt_finished.zip')
else:
    print('No finished boxes to download.')